# W&B Experiment Tracking — Real HiLiftAeroML KPI Surrogate

This notebook trains a **TinyTransolverKpi** surrogate on real HiLiftAeroML CSV summaries. The model predicts integrated aerodynamic coefficients `cl`, `cd`, and `cm` from high-lift design variables plus angle of attack.

The run demonstrates the W&B Models workflow on the real HiLiftAeroML KPI use case:

1. Pull the real KPI dataset from `wandb-registry-high_lift_surrogates/datasets:production`.
2. Train a TinyTransolver-style model on eight semantic flap/slat design tokens.
3. Train a pooled-token MLP baseline on the same split.
4. Log live metrics, prediction tables, drag-polars, and predicted-vs-true plots.
5. Promote the best TinyTransolver checkpoint to `wandb-registry-high_lift_surrogates/checkpoints:production`.
6. Save a compact W&B Workspace focused on KPI metrics and media.

**Prerequisites**
- `wandb login` from this environment.
- `registry.ipynb` executed first.
- Access to the `wandb_emea` entity on W&B, or edit your local W&B env config.


## 1. Setup

Third-party imports plus shared workshop helpers from `src/physicsx_demo/utils`.
The notebook stays focused on the W&B experiment flow; reusable data,
model, metric, plotting, and training code lives in the package.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "physicsx_demo").exists())
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import importlib

import numpy as np
import torch
import wandb

from physicsx_demo.utils.env import load_local_env, resolve_wandb_target
from physicsx_demo.utils.hilift_kpi import HiLiftKpiDataset, kpi_frame_summary, prepare_hilift_kpi_bundle


import physicsx_demo.utils.model as model_utils
model_utils = importlib.reload(model_utils)
KpiTokenMLP = model_utils.KpiTokenMLP
TinyTransolverKpi = model_utils.TinyTransolverKpi
count_params = model_utils.count_params

load_local_env()

device = torch.device("cpu")
ENTITY, PROJECT = resolve_wandb_target()

print(f"torch  = {torch.__version__}")
print(f"wandb  = {wandb.__version__}")
print(f"target = {ENTITY}/{PROJECT}")
print(f"device = {device}")


In [ ]:
# Real HiLiftAeroML KPI helpers are imported from `physicsx_demo.utils.hilift_kpi`.
# The model sees each row as 8 design-variable tokens with AoA broadcast as a condition.


## 2. Data — from the W&B Registry

`registry.ipynb` promoted the real HiLiftAeroML KPI artifact to:

`wandb-registry-high_lift_surrogates/datasets:production`

This notebook consumes that artifact and rebuilds the same held-out geometry split. The split matters: validation and test rows come from `GeoID`s not seen in training.


In [ ]:
REGISTRY_NAME = "high_lift_surrogates"
DATASET_COLLECTION = "datasets"
DATASET_ARTIFACT_NAME = "hilift_kpi_slice"
DATASET_REGISTRY_PATH = f"wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}:production"
SEED = 42
N_GEOMETRIES = 80

LOCAL_DATA_DIR = repo_root.parent / "data" / "hiliftaeroml_subset"
if not (LOCAL_DATA_DIR / "force_mom_all.csv").exists():
    LOCAL_DATA_DIR = repo_root / "data" / "hiliftaeroml_subset"

api = wandb.Api(overrides={"entity": ENTITY, "project": PROJECT})
try:
    art = api.artifact(DATASET_REGISTRY_PATH)
    ds_dir = Path(art.download())
    ACTIVE_DATASET_REF = DATASET_REGISTRY_PATH
    print(f"pulled {DATASET_REGISTRY_PATH}")
except Exception as exc:
    print(f"Registry dataset not found yet: {type(exc).__name__}: {exc}")
    print("Bootstrapping the real HiLift KPI dataset artifact from local CSVs...")
    if not (LOCAL_DATA_DIR / "force_mom_all.csv").exists() or not (LOCAL_DATA_DIR / "geo_values_all.csv").exists():
        raise FileNotFoundError(
            f"Missing local HiLiftAeroML CSVs under {LOCAL_DATA_DIR}. "
            "Run scripts/download_hilift_subset.py --include-images --execute first, or run registry.ipynb."
        ) from exc

    local_bundle = prepare_hilift_kpi_bundle(LOCAL_DATA_DIR, n_geometries=N_GEOMETRIES, seed=SEED)
    summary = kpi_frame_summary(local_bundle.frame)
    train_summary = kpi_frame_summary(local_bundle.train)
    val_summary = kpi_frame_summary(local_bundle.val)
    test_summary = kpi_frame_summary(local_bundle.test)

    preview_rows = []
    for row in local_bundle.frame.head(20).itertuples(index=False):
        preview_rows.append([row.GeoID, float(row.AoA), float(row.cl), float(row.cd), float(row.cm)])
    preview_table = wandb.Table(data=preview_rows, columns=["GeoID", "AoA", "cl", "cd", "cm"])
    split_table = wandb.Table(
        data=[
            ["all", *summary.values()],
            ["train", *train_summary.values()],
            ["val", *val_summary.values()],
            ["test", *test_summary.values()],
        ],
        columns=["split", "rows", "geometries", "aoa_min", "aoa_max"],
    )

    with wandb.init(
        entity=ENTITY,
        project=PROJECT,
        job_type="dataset-registration",
        tags=["notebook", "registry", "dataset", "real-hiliftaeroml", "bootstrap"],
        name="hilift-kpi-dataset-bootstrap",
    ) as bootstrap_run:
        dataset_artifact = wandb.Artifact(
            name=DATASET_ARTIFACT_NAME,
            type="dataset",
            description="Real HiLiftAeroML KPI CSV summaries for CL/CD/CM surrogate training.",
            metadata={
                "source": "nvidia/HiLiftAeroML",
                "n_geometries_used_for_training": N_GEOMETRIES,
                "seed": SEED,
                "summary": summary,
                "train_summary": train_summary,
                "val_summary": val_summary,
                "test_summary": test_summary,
            },
        )
        dataset_artifact.add_file(str(LOCAL_DATA_DIR / "force_mom_all.csv"), name="force_mom_all.csv")
        dataset_artifact.add_file(str(LOCAL_DATA_DIR / "geo_values_all.csv"), name="geo_values_all.csv")
        if (LOCAL_DATA_DIR / "splits").exists():
            dataset_artifact.add_dir(str(LOCAL_DATA_DIR / "splits"), name="splits")
        dataset_artifact.add(preview_table, "coefficient_preview")
        dataset_artifact.add(split_table, "split_summary")
        bootstrap_run.log({"dataset/coefficient_preview": preview_table, "dataset/split_summary": split_table})
        logged = bootstrap_run.log_artifact(dataset_artifact, aliases=["production", "real-kpi", "v1"])
        logged.wait()

        ACTIVE_DATASET_REF = f"{ENTITY}/{PROJECT}/{logged.name}:production"
        try:
            try:
                registry = api.create_registry(name=REGISTRY_NAME, visibility="organization")
                print(f"created registry: {registry.name}")
            except Exception as registry_exc:
                print(f"registry create skipped (may already exist): {registry_exc}")
            target_path = f"wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}"
            bootstrap_run.link_artifact(logged, target_path=target_path, aliases=["production", "real-kpi", "v1"])
            ACTIVE_DATASET_REF = DATASET_REGISTRY_PATH
            print(f"linked dataset -> {target_path}:production")
        except Exception as link_exc:
            print(f"Registry link skipped; using project artifact instead: {type(link_exc).__name__}: {link_exc}")
            print(f"project artifact ref: {ACTIVE_DATASET_REF}")

    art = api.artifact(ACTIVE_DATASET_REF)
    ds_dir = Path(art.download())

bundle = prepare_hilift_kpi_bundle(ds_dir, n_geometries=N_GEOMETRIES, seed=SEED)
DATASET_REGISTRY_PATH = ACTIVE_DATASET_REF

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"active dataset ref: {DATASET_REGISTRY_PATH}")
print(f"artifact dir: {ds_dir}")
print(f"all:   {kpi_frame_summary(bundle.frame)}")
print(f"train: {kpi_frame_summary(bundle.train)}")
print(f"val:   {kpi_frame_summary(bundle.val)}")
print(f"test:  {kpi_frame_summary(bundle.test)}")

sample_x, sample_y = HiLiftKpiDataset(bundle.train, bundle.stats)[0]
print()
print("One KPI row (what the model sees):")
print(f"  token features shape: {tuple(sample_x.shape)}   (8 tokens x [value, semantic_xyz, aoa_norm, sin_aoa, cos_aoa])")
print(f"  target shape        : {tuple(sample_y.shape)}   ([cl, cd, cm], normalised)")


## 3. The Model — TinyTransolverKpi

The model uses `PhysicsAttention` blocks over design-variable tokens rather than mesh points.

Each case has eight tokens:

- inboard/outboard flap deflection
- inboard/outboard flap gap multiplier
- inboard/outboard slat deflection
- inboard/outboard slat gap multiplier

The model predicts one coefficient vector by averaging token outputs. The baseline is a pooled-token MLP with no slice-level attention.


In [ ]:
# TinyTransolverKpi, KpiTokenMLP, and count_params are imported from `physicsx_demo.utils.model`.


In [ ]:
# The full training loop lives in `physicsx_demo.utils.kpi_training` so this notebook stays focused on W&B orchestration.


In [ ]:
MODEL_CONFIG = {
    "in_dim": 7,
    "out_dim": 3,
    "hidden": 64,
    "n_layers": 3,
    "n_slices": 8,
    "n_heads": 4,
    "ffn_expansion": 2,
    "dropout": 0.05,
    "use_global_context": True,
}

model = TinyTransolverKpi(**MODEL_CONFIG).to(device)
baseline = KpiTokenMLP().to(device)
print(model)
print()
print(f"TinyTransolverKpi parameters: {count_params(model):,}")
print(f"KPI token MLP parameters    : {count_params(baseline):,}")

with torch.no_grad():
    dummy_features = sample_x.unsqueeze(0)
    print(f"TinyTransolverKpi forward: in={tuple(dummy_features.shape)} -> out={tuple(model(dummy_features).shape)}")
    print(f"KpiTokenMLP forward      : in={tuple(dummy_features.shape)} -> out={tuple(baseline(dummy_features).shape)}")


## 4. Train the Model and Promote the Best Checkpoint

The training utility logs:

- train/validation loss curves
- physical RMSE for `cl`, `cd`, and `cm`
- TinyTransolver vs pooled-token MLP baseline metrics
- prediction tables
- predicted-vs-true plots
- drag polar comparisons
- the best model checkpoint linked to the Registry


In [ ]:
# Training, checkpointing, media logging, and Registry promotion live in `physicsx_demo.utils.kpi_training`.


In [ ]:
import physicsx_demo.utils.kpi_training as kpi_training_utils

kpi_training_utils = importlib.reload(kpi_training_utils)
run_kpi_training_experiment = kpi_training_utils.run_kpi_training_experiment

TRAIN_CONFIG = run_kpi_training_experiment(
    entity=ENTITY,
    project=PROJECT,
    registry_name=REGISTRY_NAME,
    dataset_registry_path=DATASET_REGISTRY_PATH,
    dataset_dir=ds_dir,
    n_geometries=N_GEOMETRIES,
    seed=SEED,
    device=device,
    epochs=80,
    batch_size=64,
    lr=3e-3,
)


## 5. Curate the W&B Workspace

The training run now logs a visual storyboard: dataset aero trends, design-variable heatmaps, the semantic token map, predicted-vs-true plots, drag polars, coefficient curves, residual histograms, and held-out geometry error heatmaps.


In [ ]:
import wandb_workspaces.reports.v2 as wr
import wandb_workspaces.workspaces as ws

workspace = ws.Workspace(
    entity=ENTITY,
    project=PROJECT,
    name="PhysicsX Workshop - Real HiLift KPI",
    settings=ws.WorkspaceSettings(
        x_axis="epoch",
        smoothing_type="exponential",
        smoothing_weight=20,
        max_runs=12,
    ),
    runset_settings=ws.RunsetSettings(
        query="tiny-transolver-kpi",
        pinned_columns=[
            "State",
            "Created",
            "summary.model/best_selection_score",
            "summary.baseline/best_selection_score",
            "summary.test/mse",
        ],
    ),
    auto_generate_panels=False,
    sections=[
        ws.Section(
            name="Dataset Visuals",
            is_open=True,
            panels=[
                wr.MediaBrowser(title="Coefficient trends", media_keys=["dataset/coefficient_trends"], num_columns=1, mode="gallery"),
                wr.MediaBrowser(title="Design variables", media_keys=["dataset/design_variable_heatmap"], num_columns=1, mode="gallery"),
                wr.MediaBrowser(title="Semantic tokens", media_keys=["dataset/semantic_token_map"], num_columns=1, mode="gallery"),
                wr.MediaBrowser(title="High-lift profiles", media_keys=["dataset/high_lift_profile_gallery"], num_columns=1, mode="gallery"),
                wr.MediaBrowser(title="Profile matrix", media_keys=["dataset/high_lift_profile_matrix"], num_columns=1, mode="gallery"),
                wr.MediaBrowser(title="Profile overlay", media_keys=["dataset/high_lift_profile_overlay"], num_columns=1, mode="gallery"),
                wr.MediaBrowser(title="Featured profile card", media_keys=["dataset/featured_profile_card"], num_columns=1, mode="gallery"),
                wr.MediaBrowser(title="Profile flythrough video", media_keys=["dataset/profile_flythrough_video"], num_columns=1, mode="gallery"),
            ],
        ),
        ws.Section(
            name="Dataset Tables",
            is_open=True,
            panels=[
                wr.WeavePanelSummaryTable(table_name="dataset/preview_table"),
                wr.WeavePanelSummaryTable(table_name="dataset/design_variable_table"),
                wr.WeavePanelSummaryTable(table_name="dataset/geometry_summary_table"),
                wr.WeavePanelSummaryTable(table_name="dataset/profile_card_table"),
            ],
        ),
        ws.Section(
            name="Model vs Baseline Comparisons",
            is_open=True,
            panels=[
                wr.MediaBrowser(title="Comparison scatter", media_keys=["val/model_baseline_scatter_latest"], num_columns=1, mode="gallery"),
                wr.MediaBrowser(title="Comparison drag polar", media_keys=["val/model_baseline_drag_polar_latest"], num_columns=1, mode="gallery"),
                wr.MediaBrowser(title="Comparison coefficient curves", media_keys=["val/model_baseline_curves_latest"], num_columns=1, mode="gallery"),
                wr.MediaBrowser(title="Model minus baseline error", media_keys=["val/model_baseline_error_delta_latest"], num_columns=1, mode="gallery"),
            ],
        ),
        ws.Section(
            name="Prediction Evolution Videos",
            is_open=True,
            panels=[
                wr.MediaBrowser(title="Coefficient curve evolution", media_keys=["val/model_baseline_curves_video"], num_columns=1, mode="gallery"),
                wr.MediaBrowser(title="Error delta evolution", media_keys=["val/model_baseline_error_delta_video"], num_columns=1, mode="gallery"),
                wr.MediaBrowser(title="Drag polar evolution", media_keys=["val/drag_polar_evolution_video"], num_columns=1, mode="gallery"),
            ],
        ),
        ws.Section(
            name="KPI RMSE",
            is_open=True,
            panels=[
                wr.LinePlot(title="CL RMSE", x="epoch", y=["val/cl_rmse", "baseline/val_cl_rmse"]),
                wr.LinePlot(title="CD RMSE", x="epoch", y=["val/cd_rmse", "baseline/val_cd_rmse"]),
                wr.LinePlot(title="CM RMSE", x="epoch", y=["val/cm_rmse", "baseline/val_cm_rmse"]),
            ],
        ),
        ws.Section(
            name="Model Selection",
            is_open=True,
            panels=[
                wr.LinePlot(title="Validation MSE", x="epoch", y=["val/mse", "baseline/val_mse"]),
                wr.LinePlot(title="Model margin vs baseline", x="epoch", y=["model/selection_margin_vs_baseline"]),
            ],
        ),
        ws.Section(
            name="Profiles and Slice Attribution",
            is_open=True,
            panels=[
                wr.MediaBrowser(title="Profile-linked slice attribution video", media_keys=["model/kpi_profile_slice_video"], num_columns=1, mode="gallery"),
                wr.MediaBrowser(title="Slice attribution video", media_keys=["model/kpi_slice_attribution_video"], num_columns=1, mode="gallery"),
                wr.MediaBrowser(title="Profile-linked attribution latest", media_keys=["model/kpi_profile_slice_latest"], num_columns=1, mode="gallery"),
                wr.MediaBrowser(title="Token slice usage latest", media_keys=["model/kpi_token_slice_usage_latest"], num_columns=1, mode="gallery"),
            ],
        ),
        ws.Section(
            name="Prediction Media",
            is_open=True,
            panels=[
                wr.MediaBrowser(title="Predicted vs true", media_keys=["val/predicted_vs_true_latest"], num_columns=1, mode="gallery"),
                wr.MediaBrowser(title="Drag polar", media_keys=["val/drag_polar_latest"], num_columns=1, mode="gallery"),
                wr.MediaBrowser(title="Coefficient curves", media_keys=["val/coefficient_curves_latest"], num_columns=1, mode="gallery"),
                wr.MediaBrowser(title="Error heatmap", media_keys=["val/error_heatmap_latest"], num_columns=1, mode="gallery"),
                wr.MediaBrowser(title="Residuals", media_keys=["val/residual_hist_latest"], num_columns=1, mode="gallery"),
            ],
        ),
    ],
)

workspace.save()
WORKSPACE_URL = workspace.url
print(f"curated workspace: {WORKSPACE_URL}")


## 6. Wrap Up — What to Look at in the W&B UI

One run was created by this notebook: `tiny-transolver-kpi-train` (`job_type = train`).

**Lineage graph**
- `wandb-registry-high_lift_surrogates/datasets:production` → this training run → `wandb-registry-high_lift_surrogates/checkpoints:production`.

**Training run metrics**
- `val/mse`, `baseline/val_mse`, and `model/selection_margin_vs_baseline`.
- `val/cl_rmse`, `val/cd_rmse`, and `val/cm_rmse` in physical coefficient units.
- `test/mse` and `test/*_rmse` in run summary.

**Media**
- `dataset/coefficient_trends`, `dataset/design_variable_heatmap`, `dataset/semantic_token_map`, `dataset/high_lift_profile_gallery`, `dataset/high_lift_profile_matrix`, `dataset/high_lift_profile_overlay`, and `dataset/featured_profile_card` make the real design space visible.
- `dataset/profile_flythrough_video`, `val/model_baseline_curves_video`, `val/model_baseline_error_delta_video`, and `val/drag_polar_evolution_video` add lightweight video storyboards.
- `dataset/preview_table`, `dataset/design_variable_table`, `dataset/geometry_summary_table`, and `dataset/profile_card_table` showcase the dataset as W&B Tables.
- `val/predicted_vs_true_epoch_*` shows coefficient scatter plots.
- `val/drag_polar_epoch_*` compares predicted and true CL/CD curves.
- `val/model_baseline_scatter_epoch_*`, `val/model_baseline_drag_polar_epoch_*`, `val/model_baseline_curves_epoch_*`, and `val/model_baseline_error_delta_epoch_*` restore the side-by-side TinyTransolverKpi vs baseline comparisons.
- `val/coefficient_curves_epoch_*`, `val/error_heatmap_epoch_*`, and `val/residual_hist_epoch_*` show held-out geometry behavior.
- `model/kpi_profile_slice_video` and `model/kpi_slice_attribution_video` animate token-to-slice attribution over epoch time.
- `val/predictions_latest` and `test/predictions` are W&B Tables with per-row errors.

**Registry**
- `checkpoints/high_lift_kpi_surrogate_ckpt` is linked as `production` from the best validation epoch.
